# 실험 S: 말투를 바꿔 쓴 데이터로 학습 — 거절이 말투에 기대는가 (논문에 없는 추가)
설계·예측·판정 규칙은 `EXPERIMENT_DESIGN.md` 4.4절 (돌리기 전에 적어 둠).

같은 문장을 내용(함수·인자)은 그대로 두고 **말투만** 5가지로 바꿔 썼다 (Gemini, `src/datagen/restyle.py`).
positive 와 negative 를 같은 말투들로 바꿨으므로 말투만으로는 둘을 가를 수 없다.

| 말투 | 예 ("Call Mom at 6 PM") |
|---|---|
| howto | How do I get my phone to ring Mom at 6? |
| fragment | Mom, call, 6 PM |
| need | I really need to talk to my mom around six. |
| form | action=call; who=Mom; time=18:00 |
| nonnative | Please you make call to mother at 6 PM time. |

| 모델 | 학습 말투 (4,000개, 같은 원문) |
|---|---|
| `ours` (기존, 다시 학습 안 함) | 원문(orig) 만 |
| `S_all` | orig + 5개 전부 |
| `S_minus_howto` | howto 만 뺌 |
| `S_minus_fragment` | fragment 만 뺌 |

**두 가설** (둘 다 추정)
- 일반화: 말투를 넓혀 학습하면 **처음 보는 말투(뺀 말투)** 에서도 거절이 나아진다
- 말투 암기: 학습에 넣은 말투에서만 나아지고, 뺀 말투에서는 ours 처럼 실패한다

채점: 우리 test 400개(positive 200 + negative 200)를 orig + 5 말투로 = 약 2,400개. probs 만 생성 (출력·확률 둘 다 나온다)

런타임: **A100 GPU + 고용량 RAM (80GB)**. 🔑 Secrets 에 `HF_TOKEN`. 학습 3번 + 채점 4번이라 1시간 반 정도. 세션이 끊기면 준비 셀 두 개 → 3번 셀을 다시 실행하면 이어서 한다

## 준비 (그냥 실행)

In [ ]:
# 다른 노트북과 같은 버전 (최신 5.x 는 API 가 바뀌어 에러가 난다)
!pip -q install transformers==4.56.2 accelerate
from huggingface_hub import login
from google.colab import userdata, drive

login(userdata.get("HF_TOKEN"))
drive.mount("/content/drive")

import os
REPO = "/content/octopus_v2_reproduce"
if not os.path.exists(REPO):
    !git clone https://github.com/nahcub/octopus_v2_reproduce.git {REPO}
%cd {REPO}
!git pull
!git log --oneline -1

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import transformers, os, json, datetime; print("transformers", transformers.__version__)  # 4.56.2 여야 함

DRIVE = "/content/drive/MyDrive/octopus_v2"
EVAL = f"{DRIVE}/runs/eval/style"
TEST = "data/style/dev_styled.jsonl"
N_TEST = sum(1 for _ in open(TEST))

RUNS = ["S_all", "S_minus_howto", "S_minus_fragment"]   # 학습 파일 data/style/train_<이름>.jsonl
MODELS = {"ours": f"{DRIVE}/runs/full", **{n: f"{DRIVE}/runs/{n}" for n in RUNS}}
DELETE_AFTER_EVAL = False              # True 면 채점 끝난 모델 가중치를 Drive 에서 지운다 (probs 는 남김)

def done_training(name): return name == "ours" or os.path.exists(f"{MODELS[name]}/train_args.json")
def done_eval(name): return os.path.exists(f"{EVAL}/{name}/probs.jsonl") and sum(1 for _ in open(f"{EVAL}/{name}/probs.jsonl")) == N_TEST
print("채점 문항", N_TEST, {n: ("채점 끝" if done_eval(n) else "학습 끝" if done_training(n) else "아직") for n in MODELS})
# ours·A00 과 같은 조건(fp32 가중치 + AdamW, batch 16)은 A100 80GB 에서만 들어간다
import torch
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 2**30
assert GPU_GB > 60, f"GPU {GPU_GB:.0f} GiB: A100 80GB 가 필요하다. 런타임 연결 해제 후 A100 + 고용량 RAM 으로 다시 연결"
print(f"GPU {GPU_GB:.0f} GiB OK")

---
## 1. 데이터 확인
말투별 개수와 예시. positive·negative 가 말투를 같은 비율로 나눠 가지는지 본다.

**예측**: ours 는 6개 말투 중 어디서 positive 를 가장 많이 거절할까?
> (여기에 예측)

In [ ]:
import collections
for f in ["train_S_all", "train_S_minus_howto", "train_S_minus_fragment", "dev_styled"]:
    rows = [json.loads(l) for l in open(f"data/style/{f}.jsonl")]
    c = collections.Counter((r["kind"][:3], r["style"]) for r in rows)
    print(f, len(rows), dict(sorted(c.items())))
rows = [json.loads(l) for l in open(TEST)]
k = rows[0]["src_index"]
for r in rows:
    if r["src_index"] == k: print(f"  {r['style']:<10} {r['query']}")

## 2. 시험 학습 (30 step)
확인할 것: `[데이터]` 에 4000개, 에러 없이 끝나는지

In [ ]:
!python -m src.train --train-file data/style/train_S_all.jsonl --max-steps 30 --output-dir /content/runs/try_S

---
## 3. 본 학습 → 말투별 test 채점 (모델마다)
이미 끝난 것은 건너뛴다. `ours` 는 학습 없이 채점만.

In [ ]:
for name, path in MODELS.items():
    print(f"
==================== {name}")
    if not done_training(name):
        print("학습 시작", datetime.datetime.now())
        !python -m src.train --train-file data/style/train_{name}.jsonl --output-dir {path}
        print("학습 끝", datetime.datetime.now())
    if not done_eval(name):
        !python -m src.probs generate --model "{path}" --test-file {TEST} --out {EVAL}/{name}/probs.jsonl
    if DELETE_AFTER_EVAL and name != "ours" and done_eval(name):
        !rm -rf "{path}"/*.safetensors

---
## 4. 말투별 거절 — 핵심 표
- positive 거절률: 정상 요청을 거절한 비율 (과잉 거절, 낮을수록 좋음)
- negative 거절률: 거절해야 할 요청을 거절한 비율 (높을수록 좋음)
- `*` = 그 모델이 학습에서 뺀 말투
- 판정: 뺀 말투에서 메운 비율 = (ours − S_minus) / (ours − S_all). **0.5 이상 → 일반화, 미만 → 말투 암기**

**예측**: S_minus_howto 는 howto 말투에서 ours 와 S_all 중 어느 쪽에 가까울까?
> (여기에 예측)

In [ ]:
args = " ".join(f"--model {n} {EVAL}/{n}/probs.jsonl" for n in MODELS if done_eval(n))
!python -m src.style_report --test-file {TEST} {args} --out {EVAL}/style_report.json | tee {EVAL}/style_report.txt

---
## 끝나면
결과: Drive `runs/eval/style/style_report.txt`·`.json`, `runs/eval/style/<모델>/probs.jsonl`.
랩탑에서 Claude 에게 "말투 실험 결과 분석해줘" 라고 하면 설계 문서 4.4절 예측과 비교한다.
MASSIVE·BFCL 채점은 모든 모델이 확정된 뒤 한 번에 한다 (설계 문서 8절).